# **Section 1: Load Cleaned Dataset**


# Feature Engineering

Feature engineering is the process of transforming raw data into features that can be effectively used by machine learning algorithms.

**The objective of this notebook is to:**

- Prepare the cleaned dataset for predictive modelling.
- Convert categorical variables into numerical representations.
- Create clinically meaningful risk categories.
- Address class imbalance using SMOTE.
- Produce a model-ready dataset for machine learning.



In [1]:
# =====================================================
# STEP 1: IMPORT LIBRARIES
# =====================================================

# Pandas is used for data manipulation and analysis.
import pandas as pd

# NumPy provides numerical operations.
import numpy as np

# LabelEncoder converts text categories into numbers.
from sklearn.preprocessing import LabelEncoder

# SMOTE is used to balance the target classes.
from imblearn.over_sampling import SMOTE

In [2]:
# =====================================================
# STEP 2: LOAD CLEANED DATASET
# =====================================================

# Load the cleaned dataset produced during
# the data cleaning phase.

df = pd.read_csv("/content/drive/MyDrive/PERSONAL PROJECTS/MENTORSHIP PROJECTS/1 Stroke Risk Prediction & Clinical Insights Analysis/stroke_cleaned.csv")

# Display first 5 records.

df.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.1,never smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
# =====================================================
# CHECK DATASET SHAPE
# =====================================================

# Understanding the number of rows and columns
# helps confirm that the dataset loaded correctly.

df.shape

(5110, 11)

# **Section 2: Encode Binary Categorical Variables**


# **Binary Variable Encoding**

Machine learning algorithms require numerical inputs.

The following variables contain only two categories:

- ever_married
- Residence_type

These variables will be converted into numerical form using Label Encoding.

Example:

Yes → 1
No → 0

Urban → 1
Rural → 0

This preserves the meaning of the original categories while making them usable for modelling.

In [4]:
# =====================================================
# CREATE LABEL ENCODER
# =====================================================

# LabelEncoder converts text values into numbers.

label_encoder = LabelEncoder()

In [5]:
# =====================================================
# ENCODE EVER_MARRIED
# =====================================================

# Convert:
# Yes -> 1
# No -> 0

df["ever_married"] = label_encoder.fit_transform(
    df["ever_married"]
)

# Verify results

df["ever_married"].value_counts()

,count
ever_married,
1,3353
0,1757


In [6]:
# =====================================================
# ENCODE RESIDENCE TYPE
# =====================================================

# Convert:
# Urban -> 1
# Rural -> 0

df["Residence_type"] = label_encoder.fit_transform(
    df["Residence_type"]
)

# Verify results

df["Residence_type"].value_counts()

,count
Residence_type,
1,2596
0,2514


# **Section 3: One-Hot Encoding**

# **One-Hot Encoding**

**Some variables contain more than two categories.**

**Examples:**

**Gender:**
- Male
- Female
- Other

**Work Type:**
- Private
- Self-employed
- Govt_job
- children
- Never_worked

**Smoking Status:**
- never smoked
- formerly smoked
- smokes
- Unknown

Using Label Encoding would incorrectly imply an order.

For example:

Private = 1
Govt_job = 2
Self-employed = 3

The model may interpret 3 as greater than 1.

To avoid this issue, One-Hot Encoding is used.

In [7]:
# =====================================================
# ONE-HOT ENCODE MULTI-CATEGORY VARIABLES
# =====================================================

# Convert categorical variables into dummy variables.

df = pd.get_dummies(
    df,
    columns=[
        "gender",
        "work_type",
        "smoking_status"
    ],
    drop_first=False
)

# Verify new columns

df.head()

,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,gender_Other,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,1,1,228.69,36.6,1,False,True,False,False,False,True,False,False,False,True,False,False
1,61.0,0,0,1,0,202.21,28.1,1,True,False,False,False,False,False,True,False,False,False,True,False
2,80.0,0,1,1,0,105.92,32.5,1,False,True,False,False,False,True,False,False,False,False,True,False
3,49.0,0,0,1,1,171.23,34.4,1,True,False,False,False,False,True,False,False,False,False,False,True
4,79.0,1,0,1,0,174.12,24.0,1,True,False,False,False,False,False,True,False,False,False,True,False


# **Section 4: Create Age Band Feature**

# **Age Band Feature**

EDA showed that age is one of the strongest predictors of stroke.

**Creating age groups makes it easier to:**

- Identify high-risk populations.
- Improve dashboard reporting.
- Potentially improve model performance.

**Age Groups:**

< 40

40–60

60+

In [9]:
# =====================================================
# CREATE AGE BAND FEATURE
# =====================================================

# Categorize patients into age groups.

df["age_band"] = pd.cut(
    df["age"],
    bins=[0, 40, 60, 100],
    labels=["< 40", "40-60", "60+"]
)

# Check results

df["age_band"].value_counts()

,count
age_band,
< 40,2244
40-60,1562
60+,1304


# **Section 5: Create Glucose Risk Tier**

# **Glucose Risk Tier**

Blood glucose levels are commonly used to assess diabetes risk.

**Categories:**

**Normal:** < 100

**Pre-Diabetic:** 100–125

**Diabetic**: > 125

This feature introduces clinical meaning into the dataset.

In [10]:
# =====================================================
# CREATE GLUCOSE RISK TIER
# =====================================================

df["glucose_risk"] = pd.cut(
    df["avg_glucose_level"],
    bins=[0, 100, 125, 1000],
    labels=[
        "Normal",
        "Pre-Diabetic",
        "Diabetic"
    ]
)

df["glucose_risk"].value_counts()

,count
glucose_risk,
Normal,3131
Diabetic,1000
Pre-Diabetic,979


# **Section 6: Create BMI Category**

# **BMI Categories**

BMI is commonly categorized into:

**Underweight:** < 18.5

**Normal:** 18.5–24.9

**Overweight:** 25–29.9

**Obese:** 30+

These categories are widely used in healthcare.

In [11]:
# =====================================================
# CREATE BMI CATEGORY
# =====================================================

df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ]
)

df["bmi_category"].value_counts()

,count
bmi_category,
Obese,1893
Overweight,1610
Normal,1258
Underweight,349


# **Section 7: Encode Newly Created Categories**


The newly created categorical features must also be converted into numerical format before modelling.

In [12]:
# =====================================================
# ONE-HOT ENCODE NEW FEATURES
# =====================================================

df = pd.get_dummies(
    df,
    columns=[
        "age_band",
        "glucose_risk",
        "bmi_category"
    ],
    drop_first=False
)

df.head()

,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,...,age_band_< 40,age_band_40-60,age_band_60+,glucose_risk_Normal,glucose_risk_Pre-Diabetic,glucose_risk_Diabetic,bmi_category_Underweight,bmi_category_Normal,bmi_category_Overweight,bmi_category_Obese
0,67.0,0,1,1,1,228.69,36.6,1,False,True,...,False,False,True,False,False,True,False,False,False,True
1,61.0,0,0,1,0,202.21,28.1,1,True,False,...,False,False,True,False,False,True,False,False,True,False
2,80.0,0,1,1,0,105.92,32.5,1,False,True,...,False,False,True,False,True,False,False,False,False,True
3,49.0,0,0,1,1,171.23,34.4,1,True,False,...,False,True,False,False,False,True,False,False,False,True
4,79.0,1,0,1,0,174.12,24.0,1,True,False,...,False,False,True,False,False,True,False,True,False,False


# **Section 8: Address Class Imbalance Using SMOTE**


# **Why SMOTE?**

The target variable is highly imbalanced.

**Before SMOTE:**

Stroke = 1 : 249 patients

Stroke = 0 : 4,861 patients

Many machine learning algorithms become biased toward the majority class.



**SMOTE (Synthetic Minority Oversampling Technique):**

- Creates synthetic stroke cases.
- Balances the dataset.
- Improves model learning.
- Helps improve Recall and F1-score.

This is especially important in healthcare because missing a true stroke patient can have serious consequences.

In [13]:
# =====================================================
# SPLIT FEATURES AND TARGET
# =====================================================

X = df.drop("stroke", axis=1)

y = df["stroke"]

In [14]:
# =====================================================
# APPLY SMOTE
# =====================================================

# Create SMOTE object

smote = SMOTE(
    random_state=42
)

# Generate balanced dataset

X_resampled, y_resampled = smote.fit_resample(
    X,
    y
)

print(y.value_counts())

print("\nAfter SMOTE:\n")

print(y_resampled.value_counts())

stroke
0    4861
1     249
Name: count, dtype: int64

After SMOTE:

stroke
1    4861
0    4861
Name: count, dtype: int64


# **Section 9: Create Final Model Dataset**




In [15]:
# =====================================================
# COMBINE RESAMPLED FEATURES AND TARGET
# =====================================================

stroke_model_ready = pd.concat(
    [
        pd.DataFrame(X_resampled),
        pd.DataFrame(y_resampled, columns=["stroke"])
    ],
    axis=1
)

stroke_model_ready.head()

,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,...,age_band_40-60,age_band_60+,glucose_risk_Normal,glucose_risk_Pre-Diabetic,glucose_risk_Diabetic,bmi_category_Underweight,bmi_category_Normal,bmi_category_Overweight,bmi_category_Obese,stroke
0,67.0,0,1,1,1,228.69,36.6,False,True,False,...,False,True,False,False,True,False,False,False,True,1
1,61.0,0,0,1,0,202.21,28.1,True,False,False,...,False,True,False,False,True,False,False,True,False,1
2,80.0,0,1,1,0,105.92,32.5,False,True,False,...,False,True,False,True,False,False,False,False,True,1
3,49.0,0,0,1,1,171.23,34.4,True,False,False,...,True,False,False,False,True,False,False,False,True,1
4,79.0,1,0,1,0,174.12,24.0,True,False,False,...,False,True,False,False,True,False,True,False,False,1


# **Section 10: Save Final Dataset**

In [17]:
# =====================================================
# SAVE MODEL-READY DATASET
# =====================================================

# Save the final dataset for machine learning.

stroke_model_ready.to_csv(
    "stroke_model_ready.csv",
    index=False
)

print(
    "Model-ready dataset saved successfully."
)


Model-ready dataset saved successfully.


False

In [18]:
# =====================================================
# DOWNLOAD THE MODEL-READY DATASET TO YOUR COMPUTER
# =====================================================

from google.colab import files

files.download('stroke_model_ready.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Feature Engineering Summary**

The following transformations were completed:

✓ Encoded binary categorical variables.

✓ One-hot encoded multi-category variables.

✓ Created age risk groups.

✓ Created glucose risk tiers.

✓ Created BMI categories.

✓ Addressed severe class imbalance using SMOTE.

✓ Generated a model-ready dataset.

The resulting dataset is now suitable for machine learning model development and evaluation.